In [11]:
import scanpy as sc
import pandas as pd
import numpy as np

adata_xeno = sc.read_h5ad("adata_all.h5ad")  # or ../data/adata_all.h5ad
adata_xeno


AnnData object with n_obs × n_vars = 75467 × 41560
    obs: 'barcode_base', 'CellBarcode', 'Stages', 'CellType', 'Phenograph', 'Pseudotime_Pltr', 'Differentiation Potential_Pltr', 'Per Stage Louvain', 'CytoTRACE', 'CV', 'CCphase', 'Branch Probabilities Basal', 'Branch Probabilities Goblet', 'Branch Probabilities Ionocytes', 'Branch Probabilities Multiciliated', 'stage_from_file', 'sample', 'in_meta'

In [12]:
var_df = adata_xeno.var.reset_index().rename(columns={"index": "raw_name"})
var_df["xeno_symbol"] = var_df["raw_name"].str.split("|").str[1]

adata_xeno.var["raw_name"] = var_df["raw_name"].values
adata_xeno.var["xeno_symbol"] = var_df["xeno_symbol"].values

adata_xeno.var.head()


,raw_name,xeno_symbol
gene25011|Xelaev18004747m,gene25011|Xelaev18004747m,Xelaev18004747m
gene21250|Xetrov90028798m.L,gene21250|Xetrov90028798m.L,Xetrov90028798m.L
gene27977|Xelaev18004749m,gene27977|Xelaev18004749m,Xelaev18004749m
gene26149|Xelaev18004750m,gene26149|Xelaev18004750m,Xelaev18004750m
gene25611|Xelaev18004751m,gene25611|Xelaev18004751m,Xelaev18004751m


In [ ]:
var_df = pd.DataFrame({"raw_name": adata_xeno.var_names.astype(str)})
var_df["xeno_symbol"] = var_df["raw_name"].str.split("|", n=1).str[-1]


In [15]:
var_df["xeno_symbol"].nunique

<bound method IndexOpsMixin.nunique of 0          Xelaev18004747m
1        Xetrov90028798m.L
2          Xelaev18004749m
3          Xelaev18004750m
4          Xelaev18004751m
               ...        
41555         LOC108702355
41556          LOC398467.L
41557         LOC108703288
41558         LOC108702356
41559    Xetrov90026942m.S
Name: xeno_symbol, Length: 41560, dtype: object>

In [16]:
url = "https://download.xenbase.org/xenbase/GenePageReports/GeneGenomeHumanOrtho_xl9_2_chd.txt"
df = pd.read_csv(url, sep="\t")
df.head()

,XENBASE_GENE_ID,SYMBOL,XENBASE_GENEPAGE_ID,XENBASE_TRANSCRIPT_ID,ENTREZ_GENE_ID,MODEL_NAME,NCBI_GENE_ID,DB_ID,GENOME_ASSEMBLY,ENSEMBL,HGNC_ID,ORTHOLOG_HUMAN_SYMBOL,ORTHOLOG_HUMAN_ENTREZ
0,XB-GENE-17335793,1a11.S,XB-GENEPAGE-6538771,NaN,780751.0,XL-9_2-gene38603,9_2,148,XB XL Genome 9.2-gene,NaN,NaN,NaN,NaN
1,XB-GENE-6252610,42sp43.L,XB-GENEPAGE-5898341,XM_018234134,397745.0,XL-9_2-gene472,9_2,148,XB XL Genome 9.2-gene,NaN,NaN,NaN,NaN
2,XB-GENE-5853356,42sp50.L,XB-GENEPAGE-5853279,NaN,397889.0,XL-9_2-gene16313,9_2,148,XB XL Genome 9.2-gene,NaN,NaN,NaN,NaN
3,XB-GENE-952966,a1cf.L,XB-GENEPAGE-952964,XM_018224490|XM_041568482|XM_041568481,444478.0,XL-9_2-gene40139,9_2,148,XB XL Genome 9.2-gene,NaN,24086.0,A1CF,29974
4,XB-GENE-17333199,a2m.L,XB-GENEPAGE-481724,XM_041570164,108695707.0,XL-9_2-gene7104,9_2,148,XB XL Genome 9.2-gene,NaN,7.0,A2M,2


In [31]:
# unique symbols in each
var_symbols = set(var_df["xeno_symbol"].dropna().str.lower())
ortho_symbols = set(df["SYMBOL"].dropna().str.lower())

matched = var_symbols & ortho_symbols

print(f"var_df unique symbols:   {len(var_symbols)}")
print(f"ortho unique symbols:    {len(ortho_symbols)}")
print(f"Matched:                 {len(matched)}")



var_df unique symbols:   41187
ortho unique symbols:    22980
Matched:                 15984


In [ ]:
matched_df = var_df[var_df["xeno_symbol"].str.lower().isin(matched)].copy()
matched_df = matched_df.merge(df[["SYMBOL", "ORTHOLOG_HUMAN_SYMBOL"]], left_on="xeno_symbol", right_on="SYMBOL", how="left") # change this for the column names
matched_df.head()


,raw_name,xeno_symbol,SYMBOL,ORTHOLOG_HUMAN_SYMBOL
0,gene40437|suclg1.S,suclg1.S,suclg1.S,SUCLG1
1,gene3136|adra1d.L,adra1d.L,adra1d.L,ADRA1D
2,gene65|smox.L,smox.L,smox.L,SMOX
3,gene9327|rnf24.L,rnf24.L,rnf24.L,RNF24
4,gene16290|gnrh2.L,gnrh2.L,gnrh2.L,GNRH2


In [33]:
print(matched_df[['ORTHOLOG_HUMAN_SYMBOL','xeno_symbol']])

      ORTHOLOG_HUMAN_SYMBOL xeno_symbol
0                    SUCLG1    suclg1.S
1                    ADRA1D    adra1d.L
2                      SMOX      smox.L
3                     RNF24     rnf24.L
4                     GNRH2     gnrh2.L
...                     ...         ...
16247                 PPIL3     ppil3.S
16248                NIF3L1    nif3l1.S
16249                   NaN    xepsin.S
16250                RNF112    rnf112.S
16251                 MFAP4     mfap4.S

[16252 rows x 2 columns]


In [3]:
adata_xeno.write("../adata_xeno_with_symbols.h5ad")
